In [88]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import json

In [36]:
load_dotenv(override=True)
api_key=os.getenv("OPENAI_API_KEY")
if api_key:
    print(f"API key exists and starts with {api_key[:8]}")
else:
    print("API key doesnot exists")

MODEL = "gpt-4.1-mini"
openai = OpenAI()
    

API key exists and starts with sk-proj-


In [40]:

system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""


In [41]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7889
* To create a public link, set `share=True` in `launch()`.


In [90]:
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}
def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price=ticket_prices.get(destination_city.lower(),"Unknown Ticket Price")
    return f"The price of a ticket to the City {destination_city} is {price}"


In [53]:
get_ticket_price("xyz")

Tool called for city xyz


'The price of a ticket to the City xyz is Unknown Ticket Price'

In [54]:
#There is a particular JSON structure that is required to describe our function
price_function={
    "name":"get_ticket_price",
    "description":"Get the price of a return ticket to the destination city.",
    "parameters":{
        "type":"object",
        "properties":{
        "destination_city":{
            "type":"string",
            "description": "The name of the destination city to look up the ticket price."
        }
        },
        "required": ["destination_city"]
    }
}

  

In [60]:
tools=[{"type":"function","function":price_function}]

In [56]:
tools

{'type': 'function',
 'function': {'name': 'get_ticket_price',
  'description': 'Get the price of a return ticket to the destination city.',
  'parameters': {'type': 'object',
   'properties': {'destination_city': {'type': 'string',
     'description': 'The name of the destination city to look up the ticket price.'}},
   'required': ['destination_city']}}}

In [70]:
def handle_tool_call(message):
    tool_call=message.tool_calls[0]
    if tool_call.function.name=="get_ticket_price":
        arguments=json.loads(tool_call.function.arguments)
        print("Raw arguments:", tool_call.function.arguments)
        city=arguments.get('destination_city')
        price_details=get_ticket_price(city)
        response={
            "role":"tool",
            "content":price_details,
            "tool_call_id":tool_call.id
        }
    return response
        
    
    

In [97]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [98]:
def chat(message,history):
    history=[{"role":h["role"],"content":h["content"]} for h in history]
    messages=[{"role":"system","content":system_message}] + history + [{"role":"user","content":message}]
    response=openai.chat.completions.create(model=MODEL,messages=messages,tools=tools)
    if response.choices[0].finish_reason=="tool_calls":
        message=response.choices[0].message
        response=handle_tool_calls(message)
        messages.append(message)
        messages.append(response)
        for message in messages:
            print(message)
        response=openai.chat.completions.create(model=MODEL,messages=messages)
    return response.choices[0].message.content
        





In [99]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7911
* To create a public link, set `share=True` in `launch()`.


Tool called for city London
Tool called for city Tokyo
{'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know the answer, say so.\n"}
{'role': 'user', 'content': 'which price is chaper london or tokyo'}
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_Ha1he3kgXI4Kp9wIRysOVidp', function=Function(arguments='{"destination_city": "London"}', name='get_ticket_price'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_GEQFectEibtmgLnc2CJP2SZ3', function=Function(arguments='{"destination_city": "Tokyo"}', name='get_ticket_price'), type='function')])
[{'role': 'tool', 'content': 'The price of a ticket to the City London is $799', 'tool_call_id': 'call_Ha1he3kgXI4Kp9wIRysOVidp'}, {'role': 'tool', 'content': 'The price of a 

Traceback (most recent call last):
  File "C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\gradio\queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\gradio\route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\gradio\blocks.py", line 2157, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\gradio\blocks.py", line 1632, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\grad

In [100]:
#call tool multiple times
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [101]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7912
* To create a public link, set `share=True` in `launch()`.


Tool called for city London
Tool called for city London
Tool called for city Paris


In [103]:
#connection to DB
import sqlite3


In [114]:
import sqlite3

DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS prices (
        city TEXT PRIMARY KEY,
        price TEXT
    )
    """)
    # Insert some sample data
    cursor.execute("INSERT OR REPLACE INTO prices (city, price) VALUES (?, ?)", ("london", "799"))
    cursor.execute("INSERT OR REPLACE INTO prices (city, price) VALUES (?, ?)", ("paris", "899"))
    cursor.execute("INSERT OR REPLACE INTO prices (city, price) VALUES (?, ?)", ("tokyo", "1400"))
    conn.commit()

In [115]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [116]:
get_ticket_price("london")

DATABASE TOOL CALLED: Getting price for london


'Ticket price to london is $799'

In [117]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [118]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [119]:
get_ticket_price("Tokyo")

DATABASE TOOL CALLED: Getting price for Tokyo


'Ticket price to Tokyo is $1420'

In [121]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7913
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for Tokyo
